In [ ]:
from datetime import datetime, timedelta

import numpy as np

np.random.seed(42)

start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
time_interval = timedelta(seconds=10)
num_steps = 180
timesteps = [start_time + i * time_interval for i in range(num_steps)]

In [19]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(0.0001), ConstantVelocity(0.0001), ConstantVelocity(0)]
)

ship_path = GroundTruthPath(
    [
        GroundTruthState(
            state_vector=np.array([0, 1, 0, 0, -10, 0]), # [x, y, vx, vy, z, vz]
            timestamp=timesteps[0],
        )
    ]
)
for k in range(1, num_steps):
    ship_path.append(
        GroundTruthState(
            state_vector=transition_model.function(
                ship_path[-1], noise=False, time_interval=time_interval
            ),
            timestamp=timesteps[k],
        )
    )

In [20]:
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(timesteps, tail_length=0.3)
plotter.plot_ground_truths([ship_path], [0, 2])
plotter.fig

In [ ]:
from ordered_set import OrderedSet

q = 0.005
transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(q), ConstantVelocity(q)]
)

cart_truths = OrderedSet()

cart_truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])
for k in range(1, num_steps):
    cart_truth.append(
        GroundTruthState(
            transition_model.function(
                cart_truth[k-1], noise=True, time_interval=time_interval
            ),
            timestamp=timesteps[k],
        )
    )
cart_truths.add(cart_truth)

0

In [ ]:
plotter.plot_ground_truths(cart_truths, [0, 2])
plotter.fig